In [ ]:
# Repository bootstrap for relocated notebooks
from pathlib import Path
import os
import sys

def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "src").exists():
            return candidate
    return start

REPO_ROOT = _find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
for _extra_path in (REPO_ROOT, REPO_ROOT / "src"):
    _extra_str = str(_extra_path)
    if _extra_str not in sys.path:
        sys.path.insert(0, _extra_str)

DATA_DIR = REPO_ROOT / "data"
MODELS_DIR = REPO_ROOT / "models"
EVAL_OUTPUT_DIR = REPO_ROOT / "eval_output"
CONFIGS_DIR = REPO_ROOT / "configs"


In [ ]:
class SimpleBatteryEnv(gym.Env):
    """
    A simplified environment for managing a battery.
    - Action: Charge/Discharge the battery.
    - Observation: The current battery level.
    - Reward: Penalize for inefficient actions.
    """
    def __init__(self):
        pass

    def reset(self, seed=None, options=None):
        pass

    def step(self, action):
        pass

    def render(self):
        # Optional: for visualization
        pass

In __init__, we define the environment's properties, like the battery's capacity, and most importantly, the action_space and observation_space. These tell the agent what kind of actions it can take and what kind of observations it will receive.

In [ ]:
class SimpleBatteryEnv(gym.Env):
    def __init__(self):
        super().__init__()
        
        # Define environment parameters
        self.battery_capacity = 10.0  # kWh
        self.max_charge_rate = 2.0    # kW
        
        # Define the Action Space
        # A single continuous action: -1 (max discharge) to +1 (max charge)
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(1,), dtype=np.float32)
        
        # Define the Observation Space
        # The battery's state of charge (SoC), from 0% to 100%
        # We'll represent this as a single value from 0.0 to 1.0
        self.observation_space = spaces.Box(low=0.0, high=1.0, shape=(1,), dtype=np.float32)
        
        # Initialize the state
        self.battery_level = 0.0
        self.current_step = 0
        self.max_steps = 24 # Simulate for 24 hours (steps)

    def reset(self, seed=None, options=None):
        # ... to be implemented ...
        pass

    def step(self, action):
        # ... to be implemented ...
        pass

The reset method is called at the beginning of every new episode. It resets the environment to an initial state and returns the first observation.

In [ ]:
class SimpleBatteryEnv(gym.Env):
    # ... (previous __init__ code) ...
    def __init__(self):
        super().__init__()
        self.battery_capacity = 10.0
        self.max_charge_rate = 2.0
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(1,), dtype=np.float32)
        self.observation_space = spaces.Box(low=0.0, high=1.0, shape=(1,), dtype=np.float32)
        self.battery_level = 0.0
        self.current_step = 0
        self.max_steps = 24

    def reset(self, seed=None, options=None):
        super().reset(seed=seed) # Necessary for reproducibility
        
        # Reset battery to a random level (e.g., between 20% and 80%)
        self.battery_level = self.np_random.uniform(low=0.2, high=0.8) * self.battery_capacity
        self.current_step = 0
        
        # Return the initial observation and an empty info dict
        observation = np.array([self.battery_level / self.battery_capacity], dtype=np.float32)
        info = {}
        return observation, info

    def step(self, action):
        # ... to be implemented ...
        pass

The step function is where the magic happens. It takes an action from the agent and calculates the consequences. It must return five things: the next_observation, the reward, and three flags: terminated, truncated, and info.

In [ ]:
class SimpleBatteryEnv(gym.Env):
    # ... (previous __init__ and reset code) ...
    def __init__(self):
        super().__init__()
        self.battery_capacity = 10.0
        self.max_charge_rate = 2.0
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(1,), dtype=np.float32)
        self.observation_space = spaces.Box(low=0.0, high=1.0, shape=(1,), dtype=np.float32)
        self.battery_level = 0.0
        self.current_step = 0
        self.max_steps = 24

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.battery_level = self.np_random.uniform(low=0.2, high=0.8) * self.battery_capacity
        self.current_step = 0
        observation = np.array([self.battery_level / self.battery_capacity], dtype=np.float32)
        info = {}
        return observation, info

    def step(self, action):
        # --- 1. Interpret the Action ---
        # The action is a value from -1 to 1. We scale it to our battery's charge rate.
        charge_energy = action[0] * self.max_charge_rate  # Energy in kWh for this step

        # --- 2. Update the Environment State ---
        # Store the old level to calculate the actual change
        previous_level = self.battery_level
        
        # Update the battery level, ensuring it stays within [0, capacity]
        self.battery_level = np.clip(self.battery_level + charge_energy, 0, self.battery_capacity)
        
        # Calculate how much energy was actually stored or discharged
        actual_charge_energy = self.battery_level - previous_level

        # --- 3. Calculate the Reward ---
        # We want to design a reward that encourages smart behavior.
        reward = 0.0
        
        # Penalize for "wasted" actions (trying to charge a full battery or discharge an empty one)
        wasted_energy = abs(charge_energy - actual_charge_energy)
        reward -= wasted_energy * 5.0  # Heavy penalty for wasting energy
        
        # Small reward for keeping the battery near full, but not overcharging
        if 0.8 <= (self.battery_level / self.battery_capacity) < 1.0:
            reward += 0.5
            
        # Small cost for existing (to encourage efficiency)
        reward -= 0.1

        # --- 4. Determine the Next Observation ---
        # The agent only needs to know the new battery level.
        next_observation = np.array([self.battery_level / self.battery_capacity], dtype=np.float32)

        # --- 5. Check for Termination Conditions ---
        self.current_step += 1
        terminated = False  # In this simple case, we don't have a "failure" state
        truncated = self.current_step >= self.max_steps # The episode ends when we run out of time

        # --- 6. Return the Standard 5-tuple ---
        info = {"wasted_energy": wasted_energy} # Extra info for debugging
        
        return next_observation, reward, terminated, truncated, info

Now, let's create an instance of our environment and see the step function in action!

In [ ]:
# Create the environment
env = SimpleBatteryEnv()

# Get the first observation
obs, info = env.reset()
print(f"Initial Observation: {obs}")

# Take a random action
random_action = env.action_space.sample()
print(f"Taking a random action: {random_action}")

# Get the result from the step function
next_obs, reward, terminated, truncated, info = env.step(random_action)

# Print the results
print("\n--- Results from env.step() ---")
print(f"Next Observation: {next_obs}")
print(f"Reward: {reward:.2f}")
print(f"Terminated: {terminated}")
print(f"Truncated: {truncated}")
print(f"Info: {info}")